# Appendix A — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Appendix A - Introduction to PyTorch** of *Build a Large Language Model (From Scratch)* by Sebastian Raschka.

> **Attribution:** Portions of the code in this notebook follow and adapt the Apache-2.0-licensed implementation accompanying Sebastian Raschka's *Build a Large Language Model (From Scratch)*.  
> Original source code: https://github.com/rasbt/LLMs-from-scratch  
> Additional annotations, experiments, explanations, and study notes were created as part of my own learning and implementation process.

### 0. Appendix A Objective

Understand the core PyTorch training workflow, from **forward pass** and **loss computation** through **backpropagation** and **parameter updates**, and understand how that workflow extends to **multi-GPU training with Distributed Data Parallel (DDP)**.


### 1. Anatomy of a PyTorch Training Loop

This notebook examines the forward pass, loss computation, backpropagation, parameter updates, and evaluation.

In [ ]:
import sys
import torch
print(sys.executable)
print("Notebook kernel is working.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

### 2. DDP Conceptual Explanation

* **Data Parallelism:** Model is replicated across GPUs. Each GPU gets the same replica of the model, but a different batch. Later, the gradients are synched/averaged.

* **Distributed Data Parallelism (DDP):** is a specific implementation of Data Parallelism in PyTorch. Synchronization of gradients happens during the backward pass, using **all-reduce** across all GPUs.

> A launcher (`torch.multiprocessing as mp`) must start multiple processes for DDP. In this script, `mp.spawn` is used.  
> DDP normally runs one process per GPU. Each process has its own Python interpreter, model replica, optimizer, and local training data.
> The spawn will automatically pass the rank which is a unique process ID.

→ During DDP setup, the process group is initialized  
→ Each process creates the model and moves its model replica to its assigned GPU.  
→ The model is wrapped in `DistributedDataParallel`. DDP registers the mechanisms needed to synchronize gradients when `loss.backward()` is called.  
→ A `DistributedSampler` gives each process a different shard of the training dataset.  
→ We loop over the epochs.  
→ At the beginning of each epoch, we call `train_loader.sampler.set_epoch(epoch)` so that shuffled samples can be ordered differently across epochs.  
→ Each process loops over its own local mini-batches.  
→ The forward pass produces logits.     
→ A loss is computed by comparing the logits with the target labels.   
→ We call optimizer.zero_grad() to prevent gradients accumulation.     
→ We call `loss.backward()` to calculate gradients from the computation graph. During this backward pass, DDP synchronizes and averages the gradients across all ranks.  
→ Each process calls `optimizer.step()`, which uses the synchronized gradients to update its local model replica. Because every process receives the same averaged gradients, the model replicas remain synchronized.  The synched gradients are used to update the model parameters in a way that minimizes the loss.  
→ After training, we call `model.eval()` and compute the training and test accuracy inside `torch.no_grad()`.   
→ Finally, the distributed process group is destroyed.  

### 3. Distributed Data Parallel — What I Want to Remember

- DDP replicates the complete model on every participating GPU.
- DDP normally runs one process per GPU.
- Each process has its own Python interpreter, model replica, optimizer, and local training data.
- `DistributedSampler` assigns each process a different shard of the dataset.
- Each process performs its own forward pass and calculates a local loss.
- During `loss.backward()`, DDP synchronizes gradients across processes using all-reduce.
- Each process calls `optimizer.step()` using the synchronized gradients, keeping the model replicas aligned.
- This implementation launches its worker processes with `torch.multiprocessing.spawn`, not `torchrun`.
- The complete runnable implementation is in [`../scripts/03_train_ddp.py`](../scripts/03_train_ddp.py).

#### DDP-specific snippets to remember:

**# Process launch**

```python
import torch.multiprocessing as mp
mp.spawn(
    main,
    args=(world_size, num_epochs),
    nprocs=world_size,
```

**# Training-specific lines**  
**# The standard training loop consists of the forward pass, loss calculation, gradient clearing, backpropagation, and parameter update.**
```python
for epoch in range(num_epochs):
    train_loader.sampler.set_epoch(epoch)
    model.train()

    for features, labels in train_loader:
        optimizer.zero_grad()

        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        optimizer.step()
```

### 4. Experiment

**Environment:** RunPod, two GPUs

**Command:**

```bash
python appendix_A/scripts/03_train_ddp.py
```

**Observation:**

Add the useful result here.

### 5. Key Definitions

- **Forward pass:** Passing input data through the model to compute predictions or logits.

- **Backpropagation:** Computing gradients of the loss with respect to model parameters.

- **Gradient:** The derivative of the loss with respect to a model parameter, indicating how that parameter should change to reduce the loss.

- **Optimizer:** An algorithm that updates model parameters using their gradients.

- **Distributed Data Parallel (DDP):** PyTorch's multi-process data-parallel training approach in which each GPU holds a model replica and gradients are synchronized across processes.

- **DistributedSampler:** A sampler that partitions a dataset so that each DDP process receives a different subset of the training data.

- **Rank:** The unique identifier of a process participating in distributed training.

- **All-reduce:** A distributed collective operation used by DDP to aggregate gradients across processes so that model replicas remain synchronized.

### 6. Q/As

- **Q: What are the core steps of a standard PyTorch training iteration?**  
  Clear gradients → forward pass → compute loss → backpropagate → update parameters.

- **Q: Why is `optimizer.zero_grad()` needed?**  
  PyTorch accumulates gradients by default, so previous gradients must usually be cleared before computing the next batch's gradients.

- **Q: What changes when moving from single-GPU training to DDP?**  
  The model is replicated across processes/GPUs, each process receives different training samples, and gradients are synchronized during the backward pass.

- **Q: Why does each DDP process call `optimizer.step()` independently?**  
  Because DDP synchronizes the gradients before the optimizer update, each process applies the same effective gradients and the model replicas remain aligned.

- **Q: Why is `sampler.set_epoch(epoch)` called at the start of every epoch?**  
  It changes the deterministic shuffle ordering across epochs while still ensuring that the distributed processes receive non-overlapping dataset shards.